In [ ]:
import openai
from typing import Any, Dict, List
from dotenv import load_dotenv
import json
from openai.types.responses import Response
import requests

load_dotenv()

In [ ]:
client = openai.OpenAI()
MODEL = "gpt-5.4-mini"
BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"


def fetch_json(path: str) -> Any:
    response = requests.get(f"{BASE_URL}{path}")
    return response.json()


def get_popular_movies() -> List[Dict]:
    return fetch_json("/movies")


def get_movie_details(movie_id: int) -> Dict:
    return fetch_json(f"/movies/{movie_id}")


def get_similar_movie(movie_id: int) -> Dict:
    return fetch_json(f"/movies/{movie_id}/similar")


function_map = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_similar_movie": get_similar_movie,
}


tools = [
    {
        "type": "function",
        "name": "get_popular_movies",
        "description": "Use this to get a list of currently popular movies. Call this when you need candidate movies for recommendations, when the user asks what is popular, or when you need movie IDs before calling other movie tools.",
    },
    {
        "type": "function",
        "name": "get_movie_details",
        "description": "Use this to get detailed information about a specific movie after you already know its movie_id. Call this when the user asks for plot, rating, genres, runtime, release information, or other details about one movie.",
        "parameters": {
            "type": "object",
            "properties": {
                "movie_id": {
                    "type": "number",
                    "description": "The numeric ID of the movie to look up. Use an ID returned by get_popular_movies or one already established in the conversation context.",
                },
            },
            "required": ["movie_id"],
        },
    },
    {
        "type": "function",
        "name": "get_similar_movie",
        "description": "Use this to get similar movies for a specific movie after you already know its movie_id. Call this when the user asks about similar movies for one movie.",
        "parameters": {
            "type": "object",
            "properties": {
                "movie_id": {
                    "type": "number",
                    "description": "The numeric ID of the movie whose similar movies you want to retrieve. Use an ID returned by get_popular_movies or one already established in the conversation context.",
                },
            },
            "required": ["movie_id"],
        },
    },
]


def append_response_items(response: Response, input_list: List[Dict]) -> None:
    input_list.extend(response.output)


def append_function_call_output(item, input_list: List[Dict]) -> None:
    function_name = item.name
    function_args = json.loads(item.arguments)
    function_to_call = function_map.get(function_name)

    if function_to_call is None:
        raise ValueError(f"Unknown function call: {function_name}")

    function_return = function_to_call(**function_args)
    print(f"{function_name} 실행 with ({function_args})")
    input_list.append(
        {
            "type": "function_call_output",
            "call_id": item.call_id,
            "output": json.dumps(function_return, ensure_ascii=False),
        }
    )


def get_function_calls(response: Response) -> List[Any]:
    return [item for item in response.output if item.type == "function_call"]


def call_ai(input_list: List[Dict]) -> List[Dict]:
    has_pending_tool_calls = True

    while has_pending_tool_calls:
        response: Response = client.responses.create(
            model=MODEL,
            input=input_list,
            tools=tools,
            instructions="너는 영화 추천 챗봇이야. 사용자의 취향을 기억하고, 이미 시청한 영화를 기억하고, 대화 기록을 기반으로 개인화된 추천을 해줘. 이미 시청한 영화는 추천하지 마.",
        )
        print("========================response=========================")
        print(f"{response.output_text}")

        append_response_items(response, input_list)
        function_calls = get_function_calls(response)

        if not function_calls:
            has_pending_tool_calls = False
            continue

        for item in function_calls:
            append_function_call_output(item, input_list)

    return input_list


def ask_ai_multiple():
    input_list = []

    while True:
        user_input = input("You: ")
        if user_input == "q":
            break
        else:
            input_list.append({"role": "user", "content": user_input})
            input_list = call_ai(input_list)

In [6]:
ask_ai_multiple()

========================response=========================

get_popular_movies 실행 with ({})
========================response=========================
요즘 인기 있는 영화들 몇 편 알려드릴게요:

1. **Your Heart Will Be Broken**  
   로맨스/드라마. 고등학생 폴리나와 학교 최고 악동 바르스의 가짜 연애에서 시작된 이야기예요.

2. **Avatar: Fire and Ash**  
   SF/판타지/어드벤처. 판도라를 배경으로 설리 가족이 새로운 위협에 맞서는 이야기예요.

3. **Hoppers**  
   애니메이션/가족/코미디/SF. 인간의 의식을 동물 로봇에 옮겨 동물 세계의 비밀을 파헤치는 내용이에요.

4. **The Super Mario Bros. Movie**  
   가족/애니메이션/어드벤처. 마리오와 루이지의 마법 세계 모험이에요.

5. **Shelter**  
   액션/범죄/스릴러. 외딴섬에서 조용히 살던 남자가 소녀를 구하면서 위험한 사건에 휘말립니다.

6. **Crime 101**  
   범죄/스릴러. 101번 프리웨이에서 벌어지는 대형 절도 사건을 둘러싼 이야기예요.

원하시면 제가 이어서  
- **장르별로 인기 영화만 추려드리거나**
- **평점 높은 순으로 정리하거나**
- **재미있는 거 / 가볍게 볼 거 / 액션 위주**로 추천해드릴게요.
========================response=========================

get_movie_details 실행 with ({'movie_id': 1327819})
========================response=========================
물론이죠. 다만 제가 방금 찾아본 영화 제목은 **Hoppers**예요.  
아래에 자세히 정리해드릴게요.

### Hoppers 정보
- **장르